# Ridewise-Churn Modeling & Evaluation
- purpose: train another model to predict which rider will churn, evaluated their performance and select the best

- we have like 10,000 riders, about 10% will churn next month, which ones?
- solution : Train a model that scores each rider 0-100% churnprobability
- High score (80% +) Urgent intervention
- Medium score (40-80%) monitor closely
- Low score (<40%) stable, maintain quality service



- Logistic regression (simple,fast, interpretble)-- Baseline model
- random forest classifier --- more robust, can handle non-linear relationship
- Xgboost ---- often best perfomance(advance model)-- handle imbalance dataset

Step 1 Setp and Load data

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_theme(style="whitegrid", font_scale=1.2)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# import our evaluation library
from sklearn.metrics import (
    roc_auc_score,f1_score, precision_score, recall_score,
    RocCurveDisplay, PrecisionRecallDisplay, confusion_matrix, precision_recall_curve
)
from xgboost import XGBClassifier


# define our system location
# define the data paths
DATA_PROCESSED = os.path.join("..", "data", "processed")
MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

features = pd.read_csv(os.path.join(DATA_PROCESSED, "features.csv")) 
segment_df = pd.read_csv(os.path.join(DATA_PROCESSED, "segment_assignment.csv"))

: 

Step 2 - Prepare our Data

In [ ]:
df = features
df.head()

In [ ]:
# Drop churn columns
# cant train a model without knowing the true label

df = features.dropna(subset=["churned"]).copy()
churn_rate = df["churned"].mean()
print(f"Riders: {len(df):,}")
print(f"churned: {df["churned"].sum():,}({churn_rate * 100:1f}%)")

In [ ]:
# 10% churned is slightly imbalanced but manageable
# <5%r >40%
# we use class_weight="balanced"

### stop 2 - Features & Train/test split

In [ ]:
# user id : identifier , not a pattern
# churned : target label (can use the answer to predict itself)
EXCLUDE = ["user_id", "churned"]


# All 29 engineered features we"d keep
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]

print(f"{len(FEATURE_COLS)} features selected")
print(FEATURE_COLS)

In [ ]:
# step 2b :  prepare the array of data
# we need X and Y

X = df[FEATURE_COLS].values
y = df["churned"].values

# check the shape of these dataset
print(X.shape)
print(y.shape)

In [ ]:
# Step 2c: Train/Test split

# test_size = 0.2 aka 20%
# 80% training data (8000 riders)
# 20 testing data (2000 riders)
# standard split ratio

# random_state = 42
# everybody get same split
# any number can work
# stratify=y
# ensure that train and test have same churn rate
# without stratify: train 12% churn, test 8% === biased evaluation

# scale_pos_weight
# ratio of negative (active) to positive (churned) examples
# used by xgboost to handle class imbalance
# formular = neg/post ---7152/848 ==8.4
# tells model "churned riders are 8.4x rearer, weigh them 8.4x more

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# scale positive weight
neg, pos = (y_train==0).sum(), (y_train == 1).sum()
scale_pos =neg/pos

print(f"Train: {len(X_train):,} Test:{len(X_test):,} | Scale_pos_weight: {scale_pos:.1f}")

### Step 2b - Logistic Regression

In [ ]:
# understand the 3 models

# model : Logistic regression (Baseline)

# image plotting riders on a graph
# X = RF combined score
# Y = churn probability

# give more weight/coeficient to churned examples during training
# formular = weight = n_samples/(n_classes * class_count)

# max_iter =500 : maximum iteration for optimization
# default is 100, sometimes is not enough (converge)

lr = LogisticRegression(class_weight="balanced", max_iter=500, random_state=42)

### Step 2c - Random Forest

In [ ]:
# random Forest (ensenble method -- collection of decision trees)
# imagine 200 experts , each looking at random subset of our features

# n_estimators=200: build 200 decision tree
# max_depth = Each tree can be max 10 levels deep
# prevent overfitting  (top deep = memorize the training data)
# n-jobs = -1 : use all CPU core for paralle training (faster) 
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1
)

### Step 2d - XGBOOST

In [ ]:
# XGBOOST
# iterative Learner(self correct itself)
# Start with a model thats dumb, predicts everyone's churn probability = 10% (average)

# Round 1
# finds riders where model is mostly got it wrong
# builds a small tree to fix those mistakes
# updates the prediction slight.

# Round 2
# check: whos still predicted wrongly
# builds another small tree to fix those mistakes
# updates the predictions again

# repeats 200 times --- final model = sum of all the 200 trees 

# n_estimators = build 200 bossting rounds(trees)
# max_depth=5 : shallower trees than random forest
# XGBOOST  use many small trees: random forest use fewer large trees
# learning_rate=0.05 : how much each tree contributes
# lower -- more conservative, might generalize the training
# slow the training (more trees)
# scale pos weight == handle imbalance
# eva_metrics : optimization metric (log loss for)


xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    scale_pos_weight=scale_pos, eval_metric="logloss", use_label_encoder=False,
    random_state=42, n_jobs=-1
)

### Step 2e - Training Process

In [ ]:
MODELS : ("Logistic Regression":lr, "Random Forest":rf, "XGBOOST":xgb)

probs, preds = {},{}
for name, model in MODELS.items():
    model.fit(X_train, y_train)
    probs[name] = model.predict_proba(X_test)[:, 1] 
    preds[name] = model.predict(X_test)

    print(f"{name:25s} | ROC AUC: {roc_auc_score(y_test, probs[name]):.4f}" 
          f" | F1: {f1_score(y_test, preds[name]):.4f}"
          )   
     